# Multimodal Fusion

This notebook implements the multimodal fusion stage for the e-commerce recommendation system.

The system combines three modality-specific embeddings:
- Image embedding I from CNN-based visual feature extraction
- Description embedding D from NLP text encoding
- Review embedding R from review representation and sentiment modeling

The initial late-fusion strategy is:

F = αI + βD + γR

where α + β + γ = 1.

This keeps the architecture simple and explainable while allowing experimental weighting for each modality.

The steps in this notebook are:
1. Load image, text, and review embeddings
2. Normalize each embedding vector
3. Apply weighted multimodal fusion
4. Store the fused representation for retrieval and ranking
5. Compare the fusion output with baseline text-only or image-only recommendations


In [ ]:
import numpy as np

# Example weighted fusion for multimodal product embeddings
alpha = 0.4
beta = 0.4
gamma = 0.2

image_embedding = np.random.randn(128).astype(np.float32)
description_embedding = np.random.randn(128).astype(np.float32)
review_embedding = np.random.randn(128).astype(np.float32)

# Normalize each vector before fusion
image_norm = image_embedding / (np.linalg.norm(image_embedding) + 1e-8)
description_norm = description_embedding / (np.linalg.norm(description_embedding) + 1e-8)
review_norm = review_embedding / (np.linalg.norm(review_embedding) + 1e-8)

fused_embedding = alpha * image_norm + beta * description_norm + gamma * review_norm

print('Alpha:', alpha)
print('Beta:', beta)
print('Gamma:', gamma)
print('Fused embedding shape:', fused_embedding.shape)
print('Fused embedding norm:', np.linalg.norm(fused_embedding))


In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
image_embeddings = np.load(
    "../data/embeddings/image_embeddings.npy"
)

text_embeddings = np.load(
    "../data/embeddings/text_embeddings.npy"
)

image_ids = pd.read_csv(
    "../data/embeddings/image_product_ids.csv"
)

text_ids = pd.read_csv(
    "../data/embeddings/text_product_ids.csv"
)

print("Image:", image_embeddings.shape)
print("Text:", text_embeddings.shape)
print("Image IDs:", len(image_ids))
print("Text IDs:", len(text_ids))

Image: (4996, 2048)
Text: (5000, 384)
Image IDs: 4996
Text IDs: 5000


In [10]:
common_ids = set(image_ids["product_id"]).intersection(
    set(text_ids["product_id"])
)

print("Common products:", len(common_ids))

Common products: 4996


In [11]:
image_map = {
    pid: i
    for i, pid in enumerate(image_ids["product_id"])
}

text_map = {
    pid: i
    for i, pid in enumerate(text_ids["product_id"])
}

In [12]:
common_ids = list(common_ids)

image_indices = [
    image_map[pid]
    for pid in common_ids
]

text_indices = [
    text_map[pid]
    for pid in common_ids
]

In [13]:
I_np = image_embeddings[
    image_indices
]

T_np = text_embeddings[
    text_indices
]

print("Aligned image:", I_np.shape)
print("Aligned text:", T_np.shape)

Aligned image: (4996, 2048)
Aligned text: (4996, 384)


In [3]:
image_projection = nn.Linear(
    2048,
    256
)

text_projection = nn.Linear(
    384,
    256
)

In [14]:
I = torch.tensor(
    I_np,
    dtype=torch.float32
)

T = torch.tensor(
    T_np,
    dtype=torch.float32
)

In [15]:
I = image_projection(I)
T = text_projection(T)

In [16]:
print("Projected image:", I.shape)
print("Projected text:", T.shape)

Projected image: torch.Size([4996, 256])
Projected text: torch.Size([4996, 256])


In [17]:
I = F.normalize(
    I,
    p=2,
    dim=1
)

T = F.normalize(
    T,
    p=2,
    dim=1
)

In [18]:
alpha = 0.5
beta = 0.5

multimodal = (
    alpha * I
    + beta * T
)

In [19]:
multimodal = F.normalize(
    multimodal,
    p=2,
    dim=1
)

In [20]:
print(
    "Multimodal shape:",
    multimodal.shape
)

Multimodal shape: torch.Size([4996, 256])


In [21]:
multimodal_embeddings = (
    multimodal
    .detach()
    .cpu()
    .numpy()
    .astype(np.float32)
)

In [22]:
np.save(
    "../data/embeddings/multimodal_embeddings.npy",
    multimodal_embeddings
)

In [23]:
pd.DataFrame({
    "product_id": common_ids
}).to_csv(
    "../data/embeddings/multimodal_product_ids.csv",
    index=False
)

In [24]:
print(
    "Saved multimodal embeddings:",
    multimodal_embeddings.shape
)

print(
    "Saved product IDs:",
    len(common_ids)
)

Saved multimodal embeddings: (4996, 256)
Saved product IDs: 4996
